In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from os import cpu_count

import ipyplot
import torch
import torchvision
from lightning import LightningModule, Trainer
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from torchvision import transforms as T

import notebook_setup  # isort:skip
from acid import conf  # isort:skip
from acid.torch.data import DatasetFromSubset  # isort:skip
from acid.torch.setup import SquareModelSetup  # isort:skip
from acid.torch.lightning_modules import SquareClassifierModule  # isort:skip
from acid.torch.logging import get_logger  # isort:skip

In [ ]:
EPOCHS = 45
BATCH_SIZE = 96
NUM_WORKERS = cpu_count() - 2
LR = None  # set to None to enable auto_lr_find
USE_WANDB_LOGGING = True

In [ ]:
dataset = torchvision.datasets.ImageFolder(root=SquareModelSetup.dataset_path)
print(f"{len(dataset)} datasets for classes {dataset.classes} found")

In [ ]:
# train / val split
train_size = int(len(dataset) // 1.2)
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])

# test split
val_size = int(len(val_dataset) // 1.2)
val_dataset, test_dataset = torch.utils.data.random_split(val_dataset, [val_size, len(val_dataset) - val_size])

# transform sets
train_dataset = DatasetFromSubset(train_dataset, SquareModelSetup.transforms["train"])
val_dataset = DatasetFromSubset(val_dataset, SquareModelSetup.transforms["val"])
test_dataset = DatasetFromSubset(test_dataset, SquareModelSetup.transforms["val"])

# loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# plot some examples
images = next(iter(train_loader))
ipyplot.plot_images([T.ToPILImage()(image) for image in images[0]], img_width=SquareModelSetup.image_size[0])

In [ ]:
auto_lr_find = LR
kwargs = {"lr": LR}
if LR is None:
    kwargs = {}
    auto_lr_find = True

model = SquareClassifierModule(train_loader, val_loader, train_loader, batch_size=BATCH_SIZE, **kwargs)

logger = get_logger("Acid Chess - Square Classification", conf.NOTEBOOKS_LOGS_DIR, USE_WANDB_LOGGING)
logger.watch(model)

trainer = Trainer(
    accelerator="auto",
    default_root_dir=conf.NOTEBOOKS_TMP_DIR,
    max_epochs=EPOCHS,
    logger=logger,
    callbacks=[
        LearningRateMonitor(logging_interval="step"),
        ModelCheckpoint(dirpath=conf.NOTEBOOKS_CHECKPOINTS_DIR, save_top_k=10, monitor="val_loss"),
    ],
    auto_lr_find=auto_lr_find,
)

if auto_lr_find is True:
    trainer.tune(model)

trainer.fit(model)

In [ ]:
trainer.test(model, test_loader)